# Mini-Projeto 7
# Deep Learning com PyTorch Para Classificação de Imagens

## 1. Instalação e Importação dos Pacotes Python

In [ ]:
!pip install -q torch torchvision torchsummary

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from io import BytesIO
from torchsummary import summary

## 2. Definindo o Dispositivo (CPU ou GPU)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Dispositivo selecionado: GPU NVIDIA (CUDA)")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Dispositivo selecionado: GPU Apple (MPS)")
else:
    device = torch.device("cpu")
    print("Dispositivo selecionado: CPU")

print(f'Usando dispositivo: {device}')

## 3. Definindo Hiperparâmetros

In [ ]:
num_epochs = 10
batch_size = 64
learning_rate = 0.001

## 4. Carregando os Dados (CIFAR-10)

In [ ]:
dsa_transformacoes = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
dsa_dataset_treino = torchvision.datasets.CIFAR10(
    root='./dados',
    train=True,
    download=True,
    transform=dsa_transformacoes
)

dsa_dataset_teste = torchvision.datasets.CIFAR10(
    root='./dados',
    train=False,
    download=True,
    transform=dsa_transformacoes
)

In [ ]:
dsa_loader_treino = torch.utils.data.DataLoader(
    dsa_dataset_treino,
    batch_size=batch_size,
    shuffle=True
)

dsa_loader_teste = torch.utils.data.DataLoader(
    dsa_dataset_teste,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

## 5. Visualizando as Imagens

In [ ]:
def imshow(img):
    img = img / 2 + 0.5
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

dataiter = iter(dsa_loader_treino)
images, labels = next(dataiter)

print('Amostra de imagens de treino:')
imshow(torchvision.utils.make_grid(images[:4]))
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

## 6. Definindo a Arquitetura da Rede Neural

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

modelo_dsa = ConvNet().to(device)
print(modelo_dsa)

## 7. Resumo do Modelo

In [ ]:
summary(modelo_dsa, (3, 32, 32))

## 8. Treinamento do Modelo

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(modelo_dsa.parameters(), lr=learning_rate)

n_total_steps = len(dsa_loader_treino)

for epoch in range(num_epochs):
    running_loss = 0.0
    
    for i, (images, labels) in enumerate(dsa_loader_treino):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = modelo_dsa(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    modelo_dsa.eval()
    with torch.no_grad():
        n_correct = 0
        n_samples = 0
        
        for val_images, val_labels in dsa_loader_teste:
            val_images = val_images.to(device)
            val_labels = val_labels.to(device)
            outputs = modelo_dsa(val_images)
            _, predicted = torch.max(outputs, 1)
            n_samples += val_labels.size(0)
            n_correct += (predicted == val_labels).sum().item()
    
    acc = 100.0 * n_correct / n_samples
    avg_loss = running_loss / n_total_steps
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Erro em Treino: {avg_loss:.4f}, Acurácia em Teste: {acc:.2f} %')

print('\nTreinamento finalizado.\n')

## 9. Avaliação do Modelo

In [ ]:
modelo_dsa.eval()

with torch.no_grad():
    n_correct = 0
    n_samples = 0
    n_class_correct = [0 for _ in range(10)]
    n_class_samples = [0 for _ in range(10)]
    
    for images, labels in dsa_loader_teste:
        images = images.to(device)
        labels = labels.to(device)
        outputs = modelo_dsa(images)
        _, predicted = torch.max(outputs, 1)
        n_samples += labels.size(0)
        n_correct += (predicted == labels).sum().item()
        
        for i in range(len(labels)):
            label = labels[i]
            pred = predicted[i]
            if (label == pred):
                n_class_correct[label] += 1
            n_class_samples[label] += 1
    
    acc_geral = 100.0 * n_correct / n_samples
    print(f'Acurácia geral do modelo na base de teste: {acc_geral:.2f} %')
    print("-" * 30)
    
    for i in range(10):
        if n_class_samples[i] > 0:
            acc_classe = 100.0 * n_class_correct[i] / n_class_samples[i]
            print(f'Acurácia da classe {classes[i]}: {acc_classe:.2f} %')
        else:
            print(f'Acurácia da classe {classes[i]}: N/A (sem amostras)')

## 10. Salvando o Modelo

In [ ]:
PATH = './modelo_dsa_mp7.pth'
torch.save(modelo_dsa.state_dict(), PATH)
print(f'Modelo salvo em: {PATH}')

## 11. Deploy e Uso do Modelo

In [ ]:
model_carregado = ConvNet().to(device)
model_carregado.load_state_dict(torch.load(PATH))
model_carregado.eval()

In [ ]:
inference_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def dsa_ia_classifica_imagem(image_path, model):
    try:
        img_pil = Image.open(image_path).convert('RGB')
    except Exception as e:
        print(f"Erro ao carregar imagem: {e}")
        return
    
    img_tensor = inference_transform(img_pil)
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = F.softmax(outputs, dim=1)
        _, predicted_idx = torch.max(outputs, 1)
    
    classe_predita = classes[predicted_idx.item()]
    confianca = torch.max(probabilities).item() * 100
    
    plt.imshow(img_pil)
    plt.title(f'Classe Prevista: {classe_predita} (Confiança: {confianca:.2f}%)')
    plt.axis('off')
    plt.show()